In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter, dataset_load

file_path = "student_performance_dataset.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "harshadapatil31/student-performance-and-study-habits-dataset",
  file_path
)
df.head()


Using Colab cache for faster access to the 'student-performance-and-study-habits-dataset' dataset.


,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [46]:
df.tail()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
995,996,Male,2.8,75.4,8.2,Masters,Yes,Yes,No,60.5,70.7,C
996,997,Male,6.7,88.4,7.1,NaN,No,Yes,No,82.2,99.5,A
997,998,Female,2.6,84.5,8.0,High School,Yes,Yes,Yes,65.2,79.2,C
998,999,Female,4.6,85.3,8.1,High School,No,No,Yes,52.2,82.2,B
999,1000,Male,3.9,77.4,6.1,Masters,Yes,No,No,45.4,70.4,C


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  1000 non-null   int64  
 1   gender                      1000 non-null   object 
 2   study_time_hours            1000 non-null   float64
 3   attendance_percent          1000 non-null   float64
 4   sleep_hours                 1000 non-null   float64
 5   parental_education          898 non-null    object 
 6   internet_access             1000 non-null   object 
 7   extracurricular_activities  1000 non-null   object 
 8   part_time_job               1000 non-null   object 
 9   previous_grade              1000 non-null   float64
 10  final_exam_score            1000 non-null   float64
 11  final_grade                 1000 non-null   object 
dtypes: float64(5), int64(1), object(6)
memory usage: 93.9+ KB


In [48]:
df.columns

Index(['student_id', 'gender', 'study_time_hours', 'attendance_percent',
       'sleep_hours', 'parental_education', 'internet_access',
       'extracurricular_activities', 'part_time_job', 'previous_grade',
       'final_exam_score', 'final_grade'],
      dtype='object')

In [49]:
df.isnull().sum()

,0
student_id,0
gender,0
study_time_hours,0
attendance_percent,0
sleep_hours,0
parental_education,102
internet_access,0
extracurricular_activities,0
part_time_job,0
previous_grade,0


In [50]:
df['parental_education'].value_counts(dropna=False)

,count
parental_education,
High School,356
Bachelors,308
Masters,184
NaN,102
PhD,50


In [51]:
df['parental_education'] = df['parental_education'].fillna('Unknown')

In [52]:
df['parental_education'].value_counts(dropna=False)

,count
parental_education,
High School,356
Bachelors,308
Masters,184
Unknown,102
PhD,50


In [53]:
df.drop('student_id',axis=1,inplace=True)

In [54]:
df.head()

,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,Male,2.6,77.5,8.0,Unknown,Yes,Yes,No,85.1,83.8,B
4,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [55]:
target = 'final_exam_score'
y = df[target]
X = df.drop(columns=target, axis=1)

# now seperate train, test, val 60, 20, 20 percent

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# ============================================================
# 1. One-hot encode categorical features
# ============================================================
categorical_cols = X_train.select_dtypes(include='object').columns

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_val = pd.get_dummies(X_val, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns - crucial for consistent feature sets after one-hot encoding
# especially if some categories are not present in all splits
train_cols = X_train.columns
X_val = X_val.reindex(columns=train_cols, fill_value=0)
X_test = X_test.reindex(columns=train_cols, fill_value=0)


# ============================================================
# 2. FIT SCALER ON TRAIN ONLY (prevents leakage)
# ============================================================

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_val_s = scaler.transform(X_val)

# ============================================================
# 3. TRAIN WITH EARLY STOPPING
# ============================================================
# Note: GBM's validation_fraction creates its own internal split from X_train.
# X_val above is for your external evaluation only, not used by early stopping.

from sklearn.ensemble import GradientBoostingRegressor # Changed to Regressor

model = GradientBoostingRegressor( # Changed to Regressor
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=5,
    validation_fraction=0.1,
    n_iter_no_change=10,
    tol=0.001,
    random_state=42
)

model.fit(X_train, y_train)


train_score = model.score(X_train_s, y_train)
val_score = model.score(X_val_s, y_val)
test_score = model.score(X_test_s, y_test)

print(f"Stopped at {model.n_estimators_} iterations (of 1000 max)")
print(f"Train: {train_score:.3f}, Val: {val_score:.3f}, Test: {test_score:.3f}")
print(f"Train-Val gap: {train_score - val_score:.3f}")
print("Healthy" if train_score - val_score < 0.05 else "Overfitting detected")


Train: 600, Val: 200, Test: 200
Stopped at 54 iterations (of 1000 max)
Train: 0.663, Val: 0.682, Test: 0.713
Train-Val gap: -0.019
Healthy


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
